# MedGemma 4B — Fine-tuning con LoRA / QLoRA (Colab)

Fine-tuning de MedGemma 4B con LoRA, usando el paquete `medgemma_glaucoma/`.

Mismos dos modos que el notebook base:
- **`classifier`** — glaucoma / no glaucoma (dataset local).
- **`descriptor`** — descripciones (Hugging Face por defecto; cambiable a tu CSV).

## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers accelerate timm hf_transfer
!pip install -q -U peft bitsandbytes trl datasets

## 2. Cargar el paquete (local o clonando el repo en Colab)

In [ ]:
# Deja el paquete `medgemma_glaucoma` importable.
#  - Local: ya esta en el repo, no hace nada.
#  - Colab: clona el repo (ajusta REPO_URL) y entra en su carpeta.
import os, sys, importlib.util

REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"  # <-- ajusta a tu repo
REPO_DIR = "utils_medgemma"

if importlib.util.find_spec("medgemma_glaucoma") is None:
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone -q {REPO_URL}")
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
    sys.path.insert(0, os.getcwd())

import medgemma_glaucoma as mg
print("paquete OK | cwd:", os.getcwd())

## 3. Autenticación en Hugging Face (modelo gated)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # descarga mas rapida

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

from huggingface_hub import login
login(token=token) if token else login()

## 4. Cargar modelo + LoRA

In [ ]:
USE_QLORA = True   # True = QLoRA (4-bit, menos VRAM); False = LoRA puro (bf16)

model, processor = mg.load_model(use_qlora=USE_QLORA, for_training=True)
model = mg.apply_lora(model, use_qlora=USE_QLORA)

## 5. Preparar el dataset (elige el modo)

In [ ]:
MODE = "classifier"   # "classifier" | "descriptor"

# Pocos datos para validar el pipeline de punta a punta.
# Para el run completo sube/quita 'limit'.
all_records = mg.load_records(mode=MODE, limit=120)
train_records, test_records = mg.train_test_split(all_records, n_test=20)
print(f"MODE={MODE} | train={len(train_records)} | test={len(test_records)}")

train_dataset = mg.build_hf_dataset(train_records)
print(train_dataset)

## 6. Collator + prueba rápida

In [ ]:
collate_fn = mg.make_collate_fn(processor)

_b = collate_fn([train_dataset[0]])
print({k: tuple(v.shape) for k, v in _b.items() if hasattr(v, "shape")})

## 7. Entrenar

In [ ]:
trainer = mg.make_trainer(model, processor, train_dataset, collate_fn,
                          use_qlora=USE_QLORA, epochs=3)
trainer.train()

## 8. Guardar el adaptador LoRA

In [ ]:
ADAPTER_DIR = "medgemma-glaucoma-lora/adapter"
trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print("Adaptador guardado en:", ADAPTER_DIR)

# (Opcional) Descargar desde Colab:
# !zip -r adapter.zip medgemma-glaucoma-lora/adapter
# from google.colab import files; files.download('adapter.zip')

## 9. Probar el modelo fine-tuneado

In [ ]:
from PIL import Image
model.config.use_cache = True
model.eval()

# Prueba unitaria sobre una imagen suelta:
image = Image.open("glaucoma.jpg").convert("RGB")
print(mg.ask_image(model, processor, image,
                   "Describe this fundus image and state whether it suggests glaucoma."))

In [ ]:
# Evaluacion sobre el split de test (accuracy en modo classifier):
mg.evaluate(model, processor, test_records, mode=MODE)

## 10. (Sesiones nuevas) Recargar el adaptador sin reentrenar

In [ ]:
# from transformers import AutoModelForImageTextToText
# from peft import PeftModel
# base, processor = mg.load_model(use_qlora=USE_QLORA)
# model = PeftModel.from_pretrained(base, ADAPTER_DIR)
# model.eval()
#
# Para fundir los adaptadores en el base (despliegue, sin peft):
# merged = model.merge_and_unload()   # no funciona si el base esta en 4-bit